# Konvoliucinių neuroninių tinklų pavyzdžiai

In [1]:
from datetime import datetime
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda:0


## Mokymo ir testavimo funkcijos

In [2]:
def seconds_to_time(seconds):
    s = int(seconds) % 60
    m = int(seconds) // 60
    if m < 1:
        return f'{s}s'
    h = m // 60
    m = m % 60
    if h < 1:
        return f'{m}m{s}s'
    return f'{h}h{m}m{s}s'

In [3]:
def train(model, loader, classes, epoch_count = 10, lr = 1e-3):
  loss_func = torch.nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr = lr)
  model.train()

  start_time = datetime.now()

  for epoch in range(epoch_count):
    loss_acum = np.array([], dtype = np.float32)

    for data in loader:
      images = data[0].to(device)
      labels = torch.nn.functional.one_hot(data[1], classes).float().to(device)

      pred = model(images)
      loss = loss_func(pred, labels)
      loss_acum = np.append(loss_acum, loss.cpu().detach().numpy())

      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    current_time = datetime.now()
    elapsed = seconds_to_time((current_time - start_time).total_seconds())
    print(f'Epoch: {epoch}, Time: {elapsed}, Loss: {np.mean(loss_acum)}')

In [4]:
def evaluate(model, loader):
  model.eval()

  correct_predictions = 0
  total_predictions = 0

  start_time = datetime.now()
  for data in loader:
    images = data[0].to(device)
    labels = data[1].to(device)

    with torch.no_grad():
      pred = model(images)
    label_pred = torch.argmax(pred, axis = 1)

    correct_predictions += torch.sum(labels == label_pred)
    total_predictions += images.shape[0]

  current_time = datetime.now()
  per_image = (current_time - start_time).total_seconds() / total_predictions
  print(f'Time: {per_image * 1000}ms, Accuracy: {correct_predictions / total_predictions}')

## Duomenų rinkiniai

### FashionMNIST

In [5]:
transforms_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transforms_test = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.FashionMNIST('fminst', train = True, download = True, transform = transforms_train)
test_dataset = torchvision.datasets.FashionMNIST('fmnist', train = False, download = True, transform = transforms_test)

num_workers = 2
batch_size = 128

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = False)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')

100%|██████████| 26.4M/26.4M [00:02<00:00, 8.87MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 141kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.62MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.9MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 9.43MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 140kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.54MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.7MB/s]

Train: 60000, Test: 10000


### Grybų duomenų rinkinys

In [6]:
!wget http://klevas.mif.vu.lt/~mif28413/gmm/fungus_dataset.zip
!unzip -q fungus_dataset.zip -d fungus

--2026-03-04 16:34:50--  http://klevas.mif.vu.lt/~mif28413/gmm/fungus_dataset.zip
Resolving klevas.mif.vu.lt (klevas.mif.vu.lt)... 193.219.42.12
Connecting to klevas.mif.vu.lt (klevas.mif.vu.lt)|193.219.42.12|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45819624 (44M) [application/zip]
Saving to: ‘fungus_dataset.zip’

fungus_dataset.zip  100%[===================>]  43.70M  12.9MB/s    in 3.4s    

2026-03-04 16:34:54 (12.9 MB/s) - ‘fungus_dataset.zip’ saved [45819624/45819624]



In [ ]:
transforms_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transforms_test = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.ImageFolder('fungus/train', transform = transforms_train)
test_dataset = torchvision.datasets.ImageFolder('fungus/valid', transform = transforms_test)

num_workers = 2
batch_size = 25

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = False)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')

## Modelių architektūros

In [8]:
#@title Daugiasluoksnis perceptronas
class MlpNet(torch.nn.Module):
  def __init__(self, in_shape, hidden_features, out_classes):
    super().__init__()
    self.hidden_layer = torch.nn.Linear(np.prod(in_shape), hidden_features)
    self.output_layer = torch.nn.Linear(hidden_features, out_classes)

  def forward(self, x):
    y = torch.nn.Flatten()(x)
    y = self.hidden_layer(y)
    y = torch.nn.ReLU()(y)
    y = self.output_layer(y)

    return y

In [12]:
#@title Paprastas konvoliucinis tinklas
class SimpleConvNet(torch.nn.Module):
  def __init__(self, in_shape, num_classes):
    super().__init__()
    self.conv1 = torch.nn.Conv2d(in_shape[0], 4, (3, 3), padding = 'same')
    self.pool1 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.conv2 = torch.nn.Conv2d(4, 8, (3, 3), padding = 'same')
    self.pool2 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.fc3 = torch.nn.Linear(8 * in_shape[1] * in_shape[2] // 4 // 4, 32)
    self.fc4 = torch.nn.Linear(32, num_classes)

    self.relu = torch.nn.ReLU()
    self.flatten = torch.nn.Flatten()

  def forward(self, x):
    y = self.relu(self.conv1(x))
    y = self.pool1(y)
    y = self.relu(self.conv2(y))
    y = self.pool2(y)
    y = self.flatten(y)
    y = self.relu(self.fc3(y))
    y = self.fc4(y)
    return y

In [16]:
#@title Didesnis konvoliucinis tinklas
class ComplexConvNet(torch.nn.Module):
  def __init__(self, in_shape, num_classes):
    super().__init__()
    self.conv1_1 = torch.nn.Conv2d(in_shape[0], 8, (3, 3), padding = 'same')
    self.conv1_2 = torch.nn.Conv2d(8, 8, (3, 3), padding = 'same')
    self.pool1 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.conv2_1 = torch.nn.Conv2d(8, 16, (3, 3), padding = 'same')
    self.conv2_2 = torch.nn.Conv2d(16, 16, (3, 3), padding = 'same')
    self.pool2 = torch.nn.MaxPool2d((2, 2), (2, 2))
    self.fc3 = torch.nn.Linear(16 * in_shape[1] * in_shape[2] // 4 // 4, 64)
    self.fc4 = torch.nn.Linear(64, num_classes)

    self.relu = torch.nn.ReLU()
    self.flatten = torch.nn.Flatten()

  def forward(self, x):
    y = torch.nn.Sequential(
        self.conv1_1,
        self.relu,
        self.conv1_2,
        self.relu,
        self.pool1,
        self.conv2_1,
        self.relu,
        self.conv2_2,
        self.relu,
        self.pool2,
        self.flatten,
        self.fc3,
        self.relu,
        self.fc4
    )(x)
    return y

## Testavimas

In [17]:
class_count = 10
# model = MlpNet(train_dataset[0][0].shape, 128, class_count).to(device)
# model = SimpleConvNet(train_dataset[0][0].shape, class_count).to(device)
model = ComplexConvNet(train_dataset[0][0].shape, class_count).to(device)
print(f'Parameter count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

train(model, train_loader, class_count, epoch_count = 10)
evaluate(model, test_loader)

Parameter count: 55,042
Epoch: 0, Time: 10s, Loss: 0.7432416081428528
Epoch: 1, Time: 20s, Loss: 0.4240453243255615
Epoch: 2, Time: 30s, Loss: 0.37186768651008606
Epoch: 3, Time: 40s, Loss: 0.33914920687675476
Epoch: 4, Time: 51s, Loss: 0.3136017918586731
Epoch: 5, Time: 1m2s, Loss: 0.29499346017837524
Epoch: 6, Time: 1m11s, Loss: 0.2788738012313843
Epoch: 7, Time: 1m22s, Loss: 0.26572486758232117
Epoch: 8, Time: 1m32s, Loss: 0.2567622661590576
Epoch: 9, Time: 1m43s, Loss: 0.24584217369556427
Time: 0.1204704ms, Accuracy: 0.8968999981880188


FashionMNIST

| Modelis | Parametrų kiekis | Tikslumas |
| - | - | - |
| MlpNet | 101 770 | 87,4% |
| SimpleConvNet | 13 242 | 87,4% |
| ComplexConvNet | 55 042 | 89,6% |